# IBKR NQ / ES 歷史 1 分鐘 K 回補（Windows Jupyter）

**v1.0｜可恢復、分合約、保留原始價格、UTC 時戳、保守 API pacing、Parquet、候選缺口稽核、Google Drive for desktop 同步。**

本 Notebook 需要與 `ibkr_history_core.py` **放在同一個資料夾**，在已登入 TWS 的 Windows 電腦本機執行。API Port `7496`，只讀，**不下單**。請勿在 Google Colab 直接使用 `127.0.0.1` 連接本機 TWS。

**資料年限注意：**IBKR 已到期超過兩年的個別期貨一般不提供歷史資料；1 分鐘資料的實際起始日仍以 IBKR 回覆為準。預設 `roll_window` 模式是為了取得最長可用的**跨季度研究期間**，每個季度合約下載其作為主力及轉倉前後的時段，而不是每一張合約從首次掛牌到到期的全部資料。若真的需要每一張合約所有歷史資料，請改 `coverage_mode="full_contract"`，時間與請求量會大幅增加。

**運行流程：**(1) 安裝 → (2) 設定 → (3) 連線 → (4) 檢視候選合約 → (5) 小測試 → (6) 手動切換完整回補 → (7) 完整性驗證 → (8) 雲端同步／後續更新。

## 1. 套件安裝（僅首次）

使用 `%pip` 可確保安裝到目前 Notebook Kernel。安裝後如需要，Restart Kernel。

In [ ]:
%pip install "ib_async>=2.1" "pandas>=2.1" "pyarrow>=15"

## 2. 編輯設定

預設先跑 `smoke_test=True`：每個標的只選最新有效季度合約，抓 3 個一天區間。完整回補前手動開啟第 6 節的 `RUN_FULL=True`。請保留 `google_drive_dir=None`，直到你找到 Google Drive for desktop **本機同步路徑**。

In [ ]:
from pathlib import Path
from ib_async import IB
from ibkr_history_core import (
    Settings, discover_quarterly_contracts, build_download_plan,
    run_backfill, load_contract_data, contract_folder,
    validate_frame, gap_report, sync_tree_to_google_drive
)

DATA_ROOT = Path.cwd() / "IBKR_Futures"
GOOGLE_DRIVE_SYNC_DIR = None  # 範例: Path(r"G:\我的雲端硬碟\00Quant_Research\IBKR_Futures")

cfg = Settings(
    symbols=("NQ", "ES"),
    host="127.0.0.1", port=7496, client_id=22,
    output_dir=str(DATA_ROOT),
    google_drive_dir=None,  # 回補期間可每完成一張合約自動同步；先測試本機
    smoke_test=True,
    smoke_requests_per_contract=3,
    coverage_mode="roll_window",  # 想下載每個合約所有可用資料，改 full_contract
    min_interval_seconds=12,
    requests_per_10_minutes=45,
    refresh_days=7,
)
print("Local dataset:", DATA_ROOT.resolve())
print("Mode:", cfg.coverage_mode, "| smoke:", cfg.smoke_test)

## 3. 連線到 TWS（僅查行情；讀取權限）

若你之前在這個 Kernel 已有有效的 `ib`，則重用連線；否則新建。不同 Kernel 或工具使用不同 `client_id`。

In [ ]:
if "ib" not in globals() or not ib.isConnected():
    ib = IB()
    await ib.connectAsync(cfg.host, cfg.port, clientId=cfg.client_id,
                          readonly=True, timeout=15)
assert ib.isConnected(), "未連上 TWS。請確認 TWS 登入、7496、唯讀 API。"
print("IBKR connected:", ib.isConnected())
print("Server version:", ib.client.serverVersion())

## 4. 預覽季度合約與回補區間

使用期貨**實際合約**（例如 `NQZ6`、`ESZ6`），不使用 `CONTFUT`，也不把不同合約價差直接當成報酬。對到期超過兩年的季度合約，會在 discovery log 記錄為排除；只保留 IBKR 成功解析的合約。

In [ ]:
contracts, discovery_log = await discover_quarterly_contracts(ib, cfg)
print("Resolved contracts:", len(contracts))
display(discovery_log.tail(20))
preview = build_download_plan(contracts, cfg)
display(preview.drop(columns=["contract"]))

## 5. 小規模測試：最新 NQ、ES 各 3 日

第一次建議只跑以下 Cell。下載時每分鐘顯示進度，資料儲存於 `IBKR_Futures/raw/標的/合約_conId/chunks/`。若因錯誤或手動停止中斷，已完成的區間會保留，下次繼續。

In [ ]:
cfg.smoke_test = True
cfg.refresh_only = False
smoke_plan, smoke_summary = await run_backfill(ib, cfg)
display(smoke_summary)

## 6. 正式執行歷史回補（需明確設為 True）

**可能需要數小時。** 會依照每張合約的日期窗口逐天回補。重新執行可透過 `checkpoint.json` 斷點續傳。API 錯誤不會被當成正常休市而跳過；會停下並在 `failures.jsonl` 記錄。請讓 TWS、Jupyter 和 Windows 保持執行。

In [ ]:
RUN_FULL = False  # 確認小測試正常後，手動修改為 True

if RUN_FULL:
    cfg.smoke_test = False
    cfg.refresh_only = False
    full_plan, full_summary = await run_backfill(ib, cfg)
    display(full_summary)
else:
    print("未啟動長時間回補；將 RUN_FULL 改成 True 後再執行。")

## 7. 查看原始資料與品質報告

嚴格檢查 OHLC、空值、重複時間、非整分鐘。分鐘間隔報告以芝加哥時間推測例行休市，**不含完整 CME 假日及提前收盤日曆**；`possible_missing_trading_bars_check_holidays` 必須再查假日或交易所時間表，不等於證實遺漏。

In [ ]:
folders = sorted((DATA_ROOT / "raw" / "NQ").glob("*/"))
if not folders:
    print("尚未下載 NQ。")
else:
    sample_folder = folders[-1]
    sample = load_contract_data(sample_folder)
    print("Sample:", sample_folder.name)
    print("Validation:", validate_frame(sample))
    display(sample.tail(5))
    display(gap_report(sample).tail(15))
    print("品質報告:", sample_folder / "reports" / "validation.json")

## 8. Google Drive 同步

先於 Windows 安裝、登入 **Google Drive for desktop**。檔案總管確認實際的「我的雲端硬碟」路徑（盤符及資料夾名稱因安裝而異），建立目的資料夾後填入下方變數。這是**本機 → Drive for desktop 本機同步資料夾**的複製；真正上傳到 Google 雲端是否完成，需由 Drive 桌面程式確認。

回補期間若希望**每完成一個合約就同步**，先設定 `cfg.google_drive_dir = str(GOOGLE_DRIVE_SYNC_DIR)`，之後執行完整回補 Cell。

注意：函式不會刪除雲端舊檔、也不會同步憑證；若手動刪除本機已完成區間，雲端不會自動同步刪除。

In [ ]:
# 範例（請改成你 Windows 實際存在的 Google Drive for desktop 資料夾）：
# GOOGLE_DRIVE_SYNC_DIR = Path(r"G:\我的雲端硬碟\00Quant_Research\IBKR_Futures")

if GOOGLE_DRIVE_SYNC_DIR is not None:
    GOOGLE_DRIVE_SYNC_DIR = Path(GOOGLE_DRIVE_SYNC_DIR)
    if not GOOGLE_DRIVE_SYNC_DIR.exists():
        raise FileNotFoundError(f"請先在檔案總管建立 Drive 目的資料夾: {GOOGLE_DRIVE_SYNC_DIR}")
    print(sync_tree_to_google_drive(DATA_ROOT, GOOGLE_DRIVE_SYNC_DIR))
    cfg.google_drive_dir = str(GOOGLE_DRIVE_SYNC_DIR)
else:
    print("尚未設定 Google Drive for desktop 路徑；資料目前只存於本機。")

## 9. 未來更新最近幾日

完整回補後，新日 K 不會自己出現；TWS 必須登入且需重新跑增量更新。這個 Cell **預設不執行**。`refresh_days=7` 會重抓最近七天以填補近期重疊，但如果停機超過七天，請把天數加大（否則可能留缺口）；時間戳去重，原始合約不變。

In [ ]:
RUN_REFRESH = False  # 未來人工啟用增量更新

if RUN_REFRESH:
    cfg.smoke_test = False
    cfg.refresh_only = True
    _, refresh_summary = await run_backfill(ib, cfg)
    display(refresh_summary)
    cfg.refresh_only = False
else:
    print("尚未啟用增量更新。")

## 10. Colab 讀取與後續研究

下載儲存後，不必保持 TWS 連線也能離線分析。Google Colab 掛載相同 Drive 位置後使用：

```python
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
from pathlib import Path
base = Path('/content/drive/MyDrive/00Quant_Research/IBKR_Futures/raw/NQ')
paths = list(base.glob('*/consolidated.parquet'))
frames = [pd.read_parquet(p) for p in paths]
# 跨合約的 frames 只是 raw 資料集合；尚未依轉倉規則調整或拼接連續報酬。
```

**研究要點：** UTC 為 canonical time；比較台灣現貨時先映射 `Asia/Taipei`，Globex 交易日／美國 DST 使用 `America/Chicago`。每個合約的 `conId`、`localSymbol`、到期日及 raw OHLC 不可丟失。連續期貨、量價轉倉和去 look-ahead 要另建獨立 pipeline。IBKR 行情的保存、同步與分享仍受帳戶／交易所資料授權條款約束。

**來源：** [IBKR Historical Data Limitations](https://interactivebrokers.github.io/tws-api/historical_limitations.html)；[IBKR Historical Bars](https://www.interactivebrokers.com/docs/tws-api/doc/md/historical-bars)；[ib_async API](https://ib-api-reloaded.github.io/ib_async/api.html)。